In [ ]:
import pandas as pd

df = pd.read_csv('Zomato Restaurant Dataset.csv')

df.dropna()
df.info()
df.head()
print(df.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

top_restoran = df.sort_values(by='Aggregate rating', ascending= False).head()
top_restoran[['Restaurant Name', 'City', 'Aggregate rating']]

plt.figure()
sns.barplot(data=top_restoran, x= 'Restaurant Name', y= 'Aggregate rating')
plt.title('Top 5 Restoran dengan rating tertinggi')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

buat nyari restoran dg rating tertinggi pake nya sort_values, trs dia pake ascendingnya False, karna mau ngurutin dari yang tertinggi, abistu diambil dari top restaurant trs [[diisi nama restoran, alamat sama rating]]. abistu mau bikin tabel pake nya barplot karna dia data kategorikal jd kl pake histogram gabisa karna bukan data kuantitatif. trs didlm sns.barplot itu di isi data, labelx sm labely.

In [ ]:
korelasi = df[['Aggregate rating', 'Average Cost for two']].corr()

plt.figure()
sns.scatterplot(data=df, x= 'Aggregate rating', y='Average Cost for two')
plt.title("Korelasi antara rating restoran dengan mean pembayaran untuk 2 orang")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

korelasi antara rating dengan rata" pembelian untuk 2 orang, berdasarkan hasil dari scatterplot didapatkan bahwa, restoran dengan rating tinggi memiliki rata" harga yang lebih mahal dibanding dengan rating yg rendah. yang dimana dapat dikatakan bahwa ada harga ada kualitas. tetapi ada juga restoran dengan rating tinggi yang memiliki harga dibawah 100k.

In [ ]:
import numpy as np

q1 = np.quantile(df['Average Cost for two'], 0.25)
q3 = np.quantile(df['Average Cost for two'], 0.75)
IQR = q3-q1

low_iqr = q1 - (1.5*IQR)
up_iqr = q3 + (1.5*IQR)

print('q1: ', q1)
print('q3: ', q3)
print('batas atas', up_iqr)
print('batas bawah', low_iqr)
df_saring= df[(df['Average Cost for two'] <= up_iqr) & (df['Average Cost for two'] >= low_iqr)] # <= 1375 itu buat ambil data di bawah ambang atas (data bersih)

plt.figure()
sns.boxplot(data=df_saring, x='Average Cost for two')
plt.xticks(rotation=45)
plt.title('boxplot deteksi outliers pada kolom Average Cost for two')
plt.tight_layout()
plt.show()

okee, ini kita cari oulier dari dataset berdasarkan kolom average cost for two. setelah identifikasi q1,q3,iqr,up_iqr dan low_iqr, kita bikin data baru buat ngambil data yang bersih. df_saring itu buat filter data bersih dari seluruh data mentah.
abistu bikin boxplot, nah cara bacanya garis di 0 itu sebagai batas bawah, trs garis tengah diangka 400 sebagai median, dan garis diangka 1000 itu sebagai bawas atas, trs titik" diluar itu adalah outlier. nah kotak biru itu sebagai mayoritas atau rata" pembayaran berada di antara 250-600.

In [ ]:
print(df['Currency'].value_counts())

exchange_rates = {
    'Indian Rupees(Rs.)': 0.012,     # 1 INR approx 0.012 USD
    'Dollar($)': 1.0,                # USD tetap 1
    'Pounds(£)': 1.27,               # 1 GBP approx 1.27 USD
    'Emirati Dirham(AED)': 0.27,     # 1 AED approx 0.27 USD
    'Brazilian Real(R$)': 0.18,      # 1 BRL approx 0.18 USD
    'Rand(R)': 0.055,                # 1 ZAR approx 0.055 USD
    'Botswana Pula(P)': 0.074,       # 1 BWP approx 0.074 USD
    'Indonesian Rupiah(IDR)': 0.000063 # 1 IDR approx 0.000063 USD (1 USD ~ 15.800 IDR)
}

def convert_usd(row):
    currency = row['Currency']
    cost = row['Average Cost for two']
    usd = exchange_rates.get(currency, 1.0)
    return cost * usd

df['Cost usd'] = df.apply(convert_usd, axis=1)
print(df[['Average Cost for two', 'Currency', 'Cost usd']].head())

q1_usd = np.quantile(df['Cost usd'], 0.25)
q3_usd = np.quantile(df['Cost usd'], 0.75)
iqr = q3_usd - q1_usd

low_usd = q1_usd - (1.5*iqr)
up_usd = q3_usd + (1.5*iqr)
df_filter = df[(df['Cost usd'] <= up_usd) & (df['Cost usd'] >= low_usd)]

print('q1 usd:',q1_usd)
print('q3 usd:',q3_usd)
print('ambang atas:',up_usd)
print('ambang bawah:',low_usd)

plt.figure()
sns.boxplot(data=df_filter, x='Cost usd')
plt.title('Deteksi outlier (usd)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



values.count() digunakan untuk melihat ada berapa jumlah data yang memakai currency(mata uang). abistu bikin data buat convert nya dulu, kaya 1 usd berapa rupiah gt dan seterusnya, abistu bikin fungsi convert_usd biar bisa diconvert semua ke usd.
trs abistu bikin kolom baru namanya Cost usd buat ngisi di dalemnya hasil convert dengan menggunakan apply.
abis bikin iqr nya, trs kita buat variabel baru buat filter data bersih dari data mentah. trs dibikin boxplot nya.
untuk boxplot masi sama dengan yg diatas, bahkan ketika diubah ke usd pun outlier yang terdeteksi masii sangat banyak.

In [ ]:
makan = df['Cuisines'].dropna().str.split(', ')
top_cuisines = makan.explode().value_counts().head(10)

plt.figure()
sns.barplot(x=top_cuisines.values, y=top_cuisines.index)
plt.title('top 10 jenis makanan terpopuler')
plt.xlabel('cuisines value (jumlah)')
plt.ylabel('cuisines index (jenis)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

langkah pertama bikin variabel baru buat ngehapus data kosong dan ngilangin spasi, abistu bikin lagi variabel baru yang diisi dengan variabl yang tadi udh dibersihin trs pake .value_counts() buat ngitung brp aja yang suka jenis makanan, trs explode itu fungsinya buat misalhin list ke baris baru.
abistu bikin diagram pake barplot, trs di isi x nya pake value karna buat nentuin jumlah nya, dan y pake index buat ngasi tau jenis nya apa aja.

In [ ]:
plt.figure()
sns.scatterplot(data=df_filter, y='Aggregate rating', x='Cost usd')
plt.title('korelasi antara rating dengan harga')
plt.ylabel('rating restoran')
plt.xlabel('harga (usd)')
plt.tight_layout()
plt.show()